# W&B Models Introduction Hands-on

This notebook walks through the key features of W&B Models using small examples.

1. Experiment Tracking
2. Tables and Rich Media
3. Artifacts
4. Sweeps
5. Registry
6. Reports
7. Extended Capabilities

Each chapter runs a Python script from the same directory. Work through the chapters
in order so Artifact names and metric names stay consistent across the materials.


## Setup

These materials use the `models_handson` directory as an uv project.
Run the following commands in a terminal. The lockfile uses W&B SDK 0.28.1.

```bash
cd models_handson
uv sync
uv run jupyter notebook
```

Set environment variables before running if you want to select a destination explicitly.
Otherwise, the W&B SDK resolves the Entity from your login.

```bash
export WANDB_ENTITY="your-team-entity"  # Optional; defaults to the logged-in Entity
export WANDB_PROJECT="wandb-models-handson"  # Optional
# Dedicated Cloud / Self-Managed only
export WANDB_BASE_URL="https://your-wandb-host"
```

Set the API key through `WANDB_API_KEY` or save it with `uv run wandb login`.
Do not hard-code API keys, Team names, or Project names in source code.


In [ ]:
# Prepare packages from a terminal.
# cd models_handson
# uv sync
# uv run jupyter notebook


In [ ]:
import os
from pathlib import Path
import runpy
import subprocess
import sys

import wandb

os.environ.setdefault("WANDB_PROJECT", "wandb-models-handson")
wandb.login()
api = wandb.Api()

# Support launching from en, models_handson, or the repository root.
candidates = [
    Path.cwd(),
    Path.cwd() / "en",
    Path.cwd() / "models_handson" / "en",
]
HANDSON_DIR = next(
    (candidate for candidate in candidates if (candidate / "1_experiment.py").exists()),
    None,
)
if HANDSON_DIR is None:
    raise RuntimeError("Could not find models_handson/en from the working directory.")
HANDSON_DIR = HANDSON_DIR.resolve()

print(f"Entity: {api.default_entity}")
print(f"Project: {os.environ['WANDB_PROJECT']}")
print(f"Scripts: {HANDSON_DIR}")


# 1. Experiment Tracking

Organize Runs with Config, Tags, Group, and Job Type, then log namespaced metrics
and a custom X-axis. The five simulated experiments use fixed seeds.

After running, inspect the following in the Workspace:

- Comparison of `train/loss`, `val/loss`, and `val/accuracy`
- `models-handson-baselines` Group
- `experiment` and `baseline` Tags
- `custom/inverse_epoch` plotted against `custom/epoch_squared`


In [ ]:
runpy.run_path(str(HANDSON_DIR / "1_experiment.py"), run_name="__main__")


# 2. Tables and Rich Media

This chapter uses NumPy-generated images and segmentation masks without large
external downloads. It logs a single image, Mask Overlay, Table, and Tables by step.

After running, open `examples/table` and `examples/progression` in the Workspace,
compare rows, and switch between steps.


In [ ]:
runpy.run_path(str(HANDSON_DIR / "2_table.py"), run_name="__main__")


# 3. Artifacts

Artifacts provide version control for data and models. This chapter runs data
generation, preprocessing, and simulated training as separate Runs and creates:

- `models-handson-dataset`
- `models-handson-processed-dataset`
- `models-handson-model`

Inputs are declared with `use_artifact()`, so Artifact Lineage displays the
dataset → processed dataset → model relationship.


In [ ]:
runpy.run_path(str(HANDSON_DIR / "3_artifacts.py"), run_name="__main__")


# 4. Sweeps

A Random Sweep explores `learning_rate`, `epochs`, and `batch_size`, optimizing
`val/accuracy`. The local Agent stops after six trials.

After running, inspect Parallel Coordinates, Parameter Importance, and each Trial.


In [ ]:
# Create a Sweep on W&B and run six Trials locally.
runpy.run_path(str(HANDSON_DIR / "4_sweeps.py"), run_name="__main__")


# 5. Registry

Registry is a central repository for managing Artifact Versions across an organization.
Linking an Artifact to a Collection does not duplicate the underlying files.

## Required preparation

1. Open [W&B Registry](https://wandb.ai/registry/).
2. Select the `models_handson` Registry. Create it in the UI if needed.
3. Select **Create collection**, name it `handson`, and set the Artifact type to `model`.
4. Confirm the Target path is `wandb-registry-models_handson/handson`.

The code does not create the Collection. Create it manually in the UI first. See
[Create a collection](https://docs.wandb.ai/models/registry/create_collection#python-sdk-beta)
for detailed instructions.

## Programmatic workflow

The next cell links chapter 3's `models-handson-model:latest` to Registry, configures
the `candidate` Alias, Tags, and Collection card, then retrieves it through Registry.

Default Target path:

- `wandb-registry-models_handson/handson`

To use another Collection, set `WANDB_REGISTRY_TARGET_PATH` before running the cell.
The destination Collection must also be created manually in the UI first.


In [ ]:
os.environ.setdefault(
    "WANDB_REGISTRY_TARGET_PATH",
    "wandb-registry-models_handson/handson",
)
runpy.run_path(str(HANDSON_DIR / "5_registry.py"), run_name="__main__")


## Registry documentation

- [Registry overview](https://docs.wandb.ai/models/registry)
- [Create a registry](https://docs.wandb.ai/models/registry/create_registry)
- [Configure registry access](https://docs.wandb.ai/models/registry/configure_registry)
- [Create a collection](https://docs.wandb.ai/models/registry/create_collection)
- [Link an artifact version](https://docs.wandb.ai/models/registry/link_version)
- [Link a Weave prompt](https://docs.wandb.ai/models/registry/link_prompt)
- [Aliases](https://docs.wandb.ai/models/registry/aliases)
- [Download an artifact](https://docs.wandb.ai/models/registry/download_use_artifact)
- [Find registry items](https://docs.wandb.ai/models/registry/search_registry)
- [Organize with tags](https://docs.wandb.ai/models/registry/organize-with-tags)
- [Collection cards](https://docs.wandb.ai/models/registry/registry_cards)
- [Lineage and audit history](https://docs.wandb.ai/models/registry/lineage)
- [Delete registry](https://docs.wandb.ai/models/registry/delete_registry)


# 6. Reports

Reports combine Run visualizations and analysis for collaboration and sharing.

## UI workflow

1. Open the Project Workspace and select **Create report**.
2. Choose the initial Panels. You can add or remove them later.
3. Edit text, headings, Panel Grids, Run sets, and charts.
4. Select **Publish to project**, then use **Share** to collaborate.

## Main options

- Content: Heading, Markdown, List, Code, Image, Media, Table of contents
- Run organization: Runset, PanelGrid, Filter, Group, RunComparer, CodeComparer
- Charts: LinePlot, BarPlot, ScatterPlot, ScalarChart, CustomChart
- Sweep analysis: ParallelCoordinatesPlot, ParameterImportancePlot
- Data/Artifacts: Summary Table, Artifact, Artifact File Panel

## Programmatic workflow

The Report and Workspace API is in Public Preview. This example uses only
LinePlot and ParameterImportancePlot, then saves the Report as a Draft.


In [ ]:
runpy.run_path(str(HANDSON_DIR / "6_report.py"), run_name="__main__")


## Reports resources

Official documentation:

- [Reports overview](https://docs.wandb.ai/models/reports)
- [Create a report](https://docs.wandb.ai/models/reports/create-a-report)
- [Edit a report](https://docs.wandb.ai/models/reports/edit-a-report)
- [Collaborate](https://docs.wandb.ai/models/reports/collaborate-on-reports)
- [Clone and export](https://docs.wandb.ai/models/reports/clone-and-export-reports)
- [Embed](https://docs.wandb.ai/models/reports/embed-reports)
- [Cross-project reports](https://docs.wandb.ai/models/reports/cross-project-reports)
- [Reports gallery](https://docs.wandb.ai/models/reports/reports-gallery)
- [Report API Quickstart](https://colab.research.google.com/github/wandb/examples/blob/master/colabs/intro/Report_API_Quickstart.ipynb)

Example Reports:

- [Nejumi LLM Leaderboard 4](https://wandb.ai/llm-leaderboard/nejumi-leaderboard4/reports/Nejumi-LLM-4--VmlldzoxMzc1OTk1MA)
- [OpenPI and W&B for Physical AI (EN)](https://wandb.ai/wandb-smle/openpi-aloha-wandb-integration/reports/OpenPI-and-W-B-for-Physical-AI-Experiment-tracking-guide--VmlldzoxNjQyMzc4Mg)
- [OpenPI x W&B (JP)](https://wandb.ai/wandb-smle/openpi-aloha-wandb-integration/reports/OpenPI-x-W-B--VmlldzoxNjQxNjczNQ)


# 7. Extended Capabilities

Use the CLI `--demo` option to try each extended capability individually.

- `resume`: append to the same Run ID
- `query`: retrieve selected metrics through the Public API
- `charts`: log a PR Curve and Audio
- `offline`: create an Offline Run and sync it later
- `alert`: send an external notification only with `--enable-alert`
- `settings`: display commonly used environment variables and Settings

The cell below runs only the default Resume demo. It never sends an Alert.


In [ ]:
subprocess.run(
    [sys.executable, str(HANDSON_DIR / "7_extended_capabilities.py"), "--demo", "resume"],
    check=True,
)


To try other demos:

```python
subprocess.run([sys.executable, str(HANDSON_DIR / "7_extended_capabilities.py"),
                "--demo", "charts"], check=True)
subprocess.run([sys.executable, str(HANDSON_DIR / "7_extended_capabilities.py"),
                "--demo", "offline"], check=True)
```

Alerts send real external notifications. Run the following only when intended.

```python
subprocess.run([sys.executable, str(HANDSON_DIR / "7_extended_capabilities.py"),
                "--demo", "alert", "--enable-alert"], check=True)
```


# Summary

- Log Config, metrics, Tables, and Media to Runs
- Version data and models with Artifacts and preserve Lineage
- Automate hyperparameter exploration with Sweeps
- Curate Artifact Versions for an organization with Registry
- Share results and analysis with Reports

Every chapter uses the same Entity resolved by the W&B SDK and the same `WANDB_PROJECT`.
Move between the Project Workspace, Artifacts, Sweeps, Registry, and Reports to inspect their relationships.
